# Stage 7: lining up the watch's axes (and calibration) with Capture-24

The models learned what Capture-24's sensor reads on a wrist: which way is its x, its y, its z. The SilverGuard watch's MPU-6050 can be mounted with its axes swapped or reversed compared with that, and Stage 6 showed a single turned axis hid most brisk walking on GOTOV.

This stage builds and tests the fix, ready for the first watch recordings:

1. **How much does a wrong orientation cost?** All 48 possibilities, on the 30 Capture-24 test people
2. **A tool that finds the orientation without labels**, and how much watch data it needs
3. **Checked on a real second device** (GOTOV)
4. **Calibration**: why the firmware's single scale factor (the watch reads 1.23 g at rest) isn't enough
5. **A full rehearsal** on a fake watch recording in the exact `record.py` format
6. **The procedure for the watch**, and the firmware code

The reusable code is in `watch_alignment.py`. **No model is trained.** **Kernel:** `.venv-tf` in this folder.

## How this works (for beginners)

**Axes.** An accelerometer reports three numbers: the push along its own x, y and z directions. Standing still, the only push is gravity (1 g), so the three numbers say which way the sensor is facing.

**Why the axes matter to the models.** Features like `x_mean`, `pitch` and `roll` (and the CNN's raw input) mean "the wrist is turned *this* way". If the watch's x is Capture-24's z, every one of those numbers means something else, and the model misreads postures.

**The 48 possibilities.** A chip mounted square in a case can only have its axes swapped and/or reversed: 6 orders (xyz, xzy, ...) times 8 sign patterns = 48. Half are real **rotations** (turning the chip). The other half are **mirror images**, which can't come from turning a chip but do come from wearing the watch on the **other wrist** (the arm's movements are mirrored).

**Finding the right one without labels.** Over a day, a wrist spends time in typical positions: flat on a desk, hanging while walking, on a pillow. Group windows by *how much* the wrist moves (this doesn't depend on axes), then look at where gravity points within each group. Turn the watch's data each of the 48 ways, and pick the one whose pattern looks most like Capture-24's. No activity labels are needed, only normal wear.

**Calibration.** Cheap sensors read slightly wrong: each axis can have an **offset** (reads +0.1 g when it should read 0) and a **gain** error (reads 1.2 g when it should read 1). Lying still in many different positions, a perfect sensor always reads exactly 1 g. Finding the offset and gain that make that true is called **sphere fitting** (all still readings should sit on a ball of radius 1 g).

**And the gyroscope?** Our models don't use it (Capture-24 has none). It sits on the same chip, so the same swap applies to it. One catch: for a **mirror image** (other wrist), rotation readings also flip sign, so gyro = −(the same matrix) × raw gyro.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

import watch_alignment as W
from common import (DATA_DIR, FEATURES, RESULTS_DIR, SEED, make_features, paired_bootstrap, scores, smooth, split_people,
                    to_binary)

f = np.load(DATA_DIR / 'features_all.npz')
F, y, pid, start = f['F'], f['y'], f['pid'], f['start']
train_ids, test_ids = split_people()
means = F[:, [FEATURES.index(k) for k in ['x_mean', 'y_mean', 'z_mean']]]
u = means / np.linalg.norm(means, axis=1, keepdims=True)       # gravity direction of each window
movement = F[:, FEATURES.index('mag_std')]                      # how much the wrist moves (doesn't depend on axes)
is_train = np.isin(pid, train_ids)

# the reference: Capture-24's training people only (the test people are what the tool is checked on)
W.save_reference(RESULTS_DIR / 'axis_reference.npz', u[is_train], movement[is_train])
reference = W.load_reference(RESULTS_DIR / 'axis_reference.npz')
print(f'{len(W.ORIENTATIONS)} candidate orientations; reference from {is_train.sum():,} windows of {len(train_ids)} people')

booster = xgb.XGBClassifier(); booster.load_model(RESULTS_DIR / 'xgb_4class.json')
hmm = np.load(RESULTS_DIR / 'hmm.npz')
# check: rotating the features directly gives the same as rotating the raw signal first
with np.load(DATA_DIR / f'{test_ids[0]}.npz') as g:
    Xc = g['X'][:200]
for i in [5, 17, 40]:
    R = W.ORIENTATIONS[i][1]
    assert np.allclose(W.rotate_features(make_features(Xc), R, FEATURES), make_features(Xc @ R.T), atol=1e-6)
print('feature rotation check passed')

48 candidate orientations; reference from 243,012 windows of 121 people


feature rotation check passed


## 1. How much does a wrong orientation cost?

XGBoost + HMM (the recommended model) on the 30 Capture-24 test people, with their data turned each of the 48 ways.

In [2]:
is_test = np.isin(pid, test_ids)
Ft, yt, pt = F[is_test], y[is_test], pid[is_test]
rows, per_person = [], np.zeros((len(W.ORIENTATIONS), len(test_ids)))
t0 = time.time()
for i, (name, R) in enumerate(W.ORIENTATIONS):
    pred = smooth(booster.predict_proba(W.rotate_features(Ft, R, FEATURES)), pt, hmm['start'], hmm['transition'])
    rows.append({'orientation': name, 'mirror (other wrist)': np.linalg.det(R) < 0,
                 'model x comes from watch': 'xyz'[int(np.flatnonzero(R[0])[0])],
                 '4-class kappa': scores(yt, pred, 4)['kappa'], 'binary kappa': scores(to_binary(yt), to_binary(pred), 2)['kappa']})
    for j, p in enumerate(test_ids):
        per_person[i, j] = scores(yt[pt == p], pred[pt == p], 4)['kappa']
cost = pd.DataFrame(rows)
cost.to_csv(RESULTS_DIR / 'stage7_orientation_cost.csv', index=False)
print(f'{time.time() - t0:.0f} s')
right = cost.loc[W.IDENTITY]
print(f'correct orientation: 4-class kappa {right["4-class kappa"]:.3f}, binary {right["binary kappa"]:.3f}')
print('\nby where the model\'s x comes from (4-class kappa):')
print(cost.groupby(['model x comes from watch', 'mirror (other wrist)'])['4-class kappa'].agg(['min', 'median', 'max']).round(3).to_string())
print('\nthe 6 worst:')
print(cost.nsmallest(6, '4-class kappa').round(3).to_string(index=False))

named = {'x and z swapped': 'model x, y, z = watch +z, +y, +x', 'x and z swapped, y reversed': 'model x, y, z = watch +z, -y, +x',
         'y and z swapped': 'model x, y, z = watch +x, +z, +y', 'y and z reversed (GOTOV)': 'model x, y, z = watch +x, -y, -z',
         'only z reversed (a mirror)': 'model x, y, z = watch +x, +y, -z'}
print('\nsome named cases:')
print(pd.DataFrame({k: cost.set_index('orientation').loc[v, ['4-class kappa', 'binary kappa']] for k, v in named.items()}).T.round(3).to_string())

33 s
correct orientation: 4-class kappa 0.818, binary 0.790

by where the model's x comes from (4-class kappa):
                                                 min  median    max
model x comes from watch mirror (other wrist)                      
x                        False                 0.665   0.761  0.818
                         True                  0.673   0.760  0.810
y                        False                 0.712   0.726  0.737
                         True                  0.712   0.730  0.744
z                        False                 0.681   0.752  0.789
                         True                  0.666   0.747  0.795

the 6 worst:
                     orientation  mirror (other wrist) model x comes from watch  4-class kappa  binary kappa
model x, y, z = watch -x, -z, -y                 False                        x          0.665         0.590
model x, y, z = watch -z, +x, +y                  True                        z          0.666         0.597
mod

**Why are some wrong orientations nearly harmless?** Capture-24's own participants didn't all wear the sensor the same way (left or right wrist, either way round), so the model has already seen some of these. Swaps that move a *different axis* into a feature's place hurt most.

## 2. The tool: finding the orientation without labels

How it decides is in the beginner section above, and the code is `W.choose_orientation`. To check it honestly, the measure is **the accuracy it costs**: for each test person, the kappa with the tool's choice versus the kappa with the correct orientation.

**Each trial:** take 3 hours of daytime wear (08:00–20:00) from 1, 3 or 5 random test people (the watch's mounting is the same for every wearer, so recordings can be pooled), let the tool choose, and measure the loss. 200 trials each.

**Two versions:**
- **All 48 candidates**
- **z kept (16 candidates):** first a 10 s check with the watch lying face up shows which watch axis goes through the face. If, as on most sensor boards, that is z, and Capture-24's z also goes through its device's face, only the 16 candidates with model z = ±watch z remain. The evidence for Capture-24's side is moderate (below), so this is an assumption, checked on GOTOV in section 3

In [3]:
still = is_train & (movement < 0.01)
axis_with_gravity = np.bincount(np.abs(u[still]).argmax(axis=1), minlength=3) / still.sum()
print('Capture-24 still windows, which axis carries most of gravity (x, y, z):', axis_with_gravity.round(2))

hour = pd.to_datetime(start).hour.to_numpy()
daytime = {p: (m := np.flatnonzero(pid == p))[(hour[m] >= 8) & (hour[m] < 20)] for p in test_ids}
rng = np.random.default_rng(SEED)
candidates = {'all 48': list(range(len(W.ORIENTATIONS))), 'z kept (16)': W.z_kept()}
rows = []
for trial in range(200):
    for n_people in [1, 3, 5]:
        chosen_people = rng.choice(test_ids, n_people, replace=False)
        idx = []
        for p in chosen_people:
            o = rng.integers(0, max(1, len(daytime[p]) - 360))
            idx.append(daytime[p][o:o + 360])                    # 360 windows of 30 s = 3 hours
        idx = np.concatenate(idx)
        d = W.rank_orientations(u[idx], movement[idx], reference)
        cols = [test_ids.index(p) for p in chosen_people]
        for cand_name, cands in candidates.items():
            pick = min(cands, key=lambda i: d[i])
            rows.append({'candidates': cand_name, 'people pooled': n_people, 'picked correct': pick == W.IDENTITY,
                         'kappa lost': np.mean(per_person[W.IDENTITY, cols] - per_person[pick, cols])})
tool = pd.DataFrame(rows)
tool.to_csv(RESULTS_DIR / 'stage7_tool_check.csv', index=False)
summary = tool.groupby(['candidates', 'people pooled']).agg(
    mean_loss=('kappa lost', 'mean'), loss_95th_percentile=('kappa lost', lambda v: np.percentile(v, 95)),
    worst_loss=('kappa lost', 'max'), share_losing_over_005=('kappa lost', lambda v: (v > 0.05).mean()),
    picked_exactly_correct=('picked correct', 'mean'))
print('\n3 hours of daytime wear per person, 200 trials each:')
print(summary.round(3).to_string())

Capture-24 still windows, which axis carries most of gravity (x, y, z): [0.36 0.15 0.49]



3 hours of daytime wear per person, 200 trials each:
                           mean_loss  loss_95th_percentile  worst_loss  share_losing_over_005  picked_exactly_correct
candidates  people pooled                                                                                            
all 48      1                  0.016                 0.183       0.293                  0.135                   0.330
            3                  0.003                 0.040       0.142                  0.045                   0.560
            5                  0.002                 0.022       0.126                  0.010                   0.690
z kept (16) 1                 -0.011                 0.023       0.150                  0.025                   0.410
            3                 -0.002                 0.016       0.053                  0.005                   0.590
            5                 -0.000                 0.016       0.040                  0.000                   0.705


A negative loss means the tool's choice scored slightly *better* than "correct" for those people: some Capture-24 people wore the sensor turned, so for them another orientation really is better.

## 3. Checked on a real second device: GOTOV

Stage 6 found GOTOV's best match was "y and z reversed". Now with the full tool, on all GOTOV windows pooled (35 people, about an hour each), and against the truth from labels: the orientation that gives the best XGBoost + HMM score.

In [4]:
s5 = np.load(RESULTS_DIR / 'stage5_gotov_predictions.npz')
Fg = np.load('processed/gotov/features.npz')['F']
gm = Fg[:, [FEATURES.index(k) for k in ['x_mean', 'y_mean', 'z_mean']]]
ug, mvg = gm / np.linalg.norm(gm, axis=1, keepdims=True), Fg[:, FEATURES.index('mag_std')]
yg, pg = s5['y'], s5['pid']
to3 = lambda v: np.maximum(np.asarray(v), 1) - 1

rows = []
for i, (name, R) in enumerate(W.ORIENTATIONS):
    pred = smooth(booster.predict_proba(W.rotate_features(Fg, R, FEATURES)), pg, hmm['start'], hmm['transition'])
    rows.append({'orientation': name, 'binary kappa': scores(to_binary(yg), to_binary(pred), 2)['kappa'],
                 '3-class kappa': scores(to3(yg), to3(pred), 3)['kappa']})
gotov_truth = pd.DataFrame(rows)
best_all, _, d_all = W.choose_orientation(ug, mvg, reference)
best_z, _, _ = W.choose_orientation(ug, mvg, reference, W.z_kept())
face = np.bincount(np.abs(ug[mvg < 0.01]).argmax(axis=1), minlength=3) / (mvg < 0.01).sum()
print('GOTOV still windows, which axis carries most of gravity (x, y, z):', face.round(2))
print('\nby labels, the best orientations for GOTOV (XGBoost + HMM):')
print(gotov_truth.nlargest(5, '3-class kappa').round(3).to_string(index=False))
print(f'\nas recorded:   {gotov_truth.loc[W.IDENTITY, "binary kappa"]:.3f} binary, {gotov_truth.loc[W.IDENTITY, "3-class kappa"]:.3f} 3-class')
for label, b in [('tool, all 48', best_all), ('tool, z kept', best_z)]:
    r = gotov_truth.loc[b]
    print(f'{label:14s} {r["orientation"]}: {r["binary kappa"]:.3f} binary, {r["3-class kappa"]:.3f} 3-class '
          f'(rank {int((gotov_truth["3-class kappa"] > r["3-class kappa"]).sum()) + 1} of 48 by labels)')

# per person: with only about an hour each, how often does the tool agree with the pooled choice?
agree = [W.choose_orientation(ug[pg == p], mvg[pg == p], reference, W.z_kept())[0] == best_z for p in np.unique(pg)]
print(f'\none person at a time (about 1 h each, z kept): same choice as pooled for {sum(agree)} of {len(agree)} people')

GOTOV still windows, which axis carries most of gravity (x, y, z): [0.4 0.1 0.5]

by labels, the best orientations for GOTOV (XGBoost + HMM):
                     orientation  binary kappa  3-class kappa
model x, y, z = watch -x, -y, -z         0.982          0.827
model x, y, z = watch +x, -y, -z         0.990          0.825
model x, y, z = watch -x, -y, +z         0.955          0.817
model x, y, z = watch +x, +y, -z         0.934          0.807
model x, y, z = watch -y, +x, -z         0.884          0.794

as recorded:   0.895 binary, 0.729 3-class
tool, all 48   model x, y, z = watch +x, -y, -z: 0.990 binary, 0.825 3-class (rank 2 of 48 by labels)
tool, z kept   model x, y, z = watch +x, -y, -z: 0.990 binary, 0.825 3-class (rank 2 of 48 by labels)



one person at a time (about 1 h each, z kept): same choice as pooled for 20 of 35 people


## 4. Calibration: offset and gain on each axis

**The firmware today** measures |a| once with the watch lying still and divides everything by it (1.23 g → 1.00 g). That is right only if every axis is off by the same factor. If instead **one axis has an offset** (a very common MPU-6050 fault), one scale factor makes things worse in other positions:

In [5]:
offset_z = np.array([0, 0, 0.23])                 # example: z reads 0.23 g too high, nothing else wrong
scale = 1 / np.linalg.norm(np.array([0, 0, 1]) + offset_z)    # the firmware's fix, measured lying face up
for label, true in [('face up', [0, 0, 1]), ('face down', [0, 0, -1]), ('arm hanging (gravity along x)', [1, 0, 0])]:
    raw = np.array(true, float) + offset_z
    print(f'{label:30s} reads {np.linalg.norm(raw):.2f} g; after the single scale factor {np.linalg.norm(raw) * scale:.2f} g (should be 1.00)')

face up                        reads 1.23 g; after the single scale factor 1.00 g (should be 1.00)
face down                      reads 0.77 g; after the single scale factor 0.63 g (should be 1.00)
arm hanging (gravity along x)  reads 1.03 g; after the single scale factor 0.83 g (should be 1.00)


**Sphere fitting** (`W.fit_calibration`) fixes offset and gain on each axis, using 10 s blocks where the watch is completely still, in several different positions. Check: distort a Capture-24 test person's data with known errors, and see if the fit recovers them. Then: how often does normal daytime wear give enough different positions?

In [6]:
true_gain = np.array([1.02, 0.97, 1.20])            # raw = true * true_gain + true_offset
true_offset = np.array([0.05, -0.03, 0.10])
with np.load(DATA_DIR / f'{test_ids[3]}.npz') as g:
    Xp = g['X']
# Capture-24's own data isn't perfectly calibrated either, so first measure this person's existing errors
own_offset, own_gain, _ = W.fit_calibration(W.still_blocks(Xp))
print("this person's own errors already in Capture-24: offset", own_offset.round(3), 'gain', (1 / own_gain).round(3))
# the fit should find the hidden errors on top of those: raw = (true / own_gain + own_offset) * true_gain + true_offset
expected_offset = true_offset + true_gain * own_offset
expected_gain = true_gain / own_gain
offset, gain, report = W.fit_calibration(W.still_blocks(Xp * true_gain + true_offset))
print(report)
print('found offset', offset.round(3), ' expected', expected_offset.round(3))
print('found gain  ', (1 / gain).round(3), ' expected', expected_gain.round(3))
cal_check = {'offset error (g)': np.abs(offset - expected_offset).max(), 'gain error': np.abs(1 / gain - expected_gain).max()}
print(cal_check)

covered = []
for p in test_ids:
    with np.load(DATA_DIR / f'{p}.npz') as g:
        hours = pd.to_datetime(g['start']).hour
        day = g['X'][(hours >= 8) & (hours < 20)]
    covered.append(W.fit_calibration(W.still_blocks(day[:360]))[2]['all axes covered'] if len(day) else False)
print(f'\n3 h of daytime wear has still positions covering all axes for {sum(covered)} of {len(covered)} test people')

gx = np.concatenate([np.load(f'processed/gotov/{p}.npz')['X'] for p in np.unique(pg)])
g_off, g_gain, g_rep = W.fit_calibration(W.still_blocks(gx))
print('\nGOTOV wrist sensors (pooled):', {k: (round(v, 1) if isinstance(v, float) else v) for k, v in g_rep.items()})

this person's own errors already in Capture-24: offset

 [ 0.005 -0.006  0.066] gain [1.001 0.993 0.939]


{'still blocks': 2993, 'all axes covered': True, 'error before (mg)': np.float64(109.06692857975663), 'error after (mg)': np.float64(3.506162252386013), 'used': True}
found offset [ 0.055 -0.036  0.179]  expected [ 0.055 -0.036  0.179]
found gain   [1.021 0.963 1.127]  expected [1.021 0.963 1.127]
{'offset error (g)': np.float64(9.68672404590698e-05), 'gain error': np.float64(6.428353092691186e-05)}



3 h of daytime wear has still positions covering all axes for 10 of 30 test people



GOTOV wrist sensors (pooled): {'still blocks': 1841, 'all axes covered': True, 'error before (mg)': np.float64(17.8), 'error after (mg)': np.float64(5.1), 'used': True}


## 5. Full rehearsal: a fake watch recording

Take 2 hours of daytime data from each of 5 test people, then secretly:
- turn the axes (model x = watch −y, model y = watch +x, model z = watch +z: the chip turned a quarter-turn on the board)
- add calibration errors (z offset +0.23 g, like this watch's 1.23 g, plus small gain errors)
- add 6 still positions at the start (the calibration routine in section 6)

and save it exactly as `firmware/FinalPrototype/record.py` does: the chip's counts (2048 per g) converted to g with 5 decimals, the `# label:` and header lines, and a row of column names. Then run the real pipeline on the file alone: load → calibrate → face-up check → choose orientation.

In [7]:
hidden = 'model x, y, z = watch -y, +x, +z'
R_hidden = dict(W.ORIENTATIONS)[hidden]
hidden_gain, hidden_offset = np.array([0.99, 1.01, 1.03]), np.array([0.01, -0.02, 0.23])
rng = np.random.default_rng(SEED + 1)
model_frame = []
for p in rng.choice(test_ids, 5, replace=False):
    with np.load(DATA_DIR / f'{p}.npz') as g:
        hours = pd.to_datetime(g['start']).hour
        day = g['X'][(hours >= 8) & (hours < 20)]
    o = rng.integers(0, max(1, len(day) - 240))
    model_frame.append(day[o:o + 240].reshape(-1, 3))           # 240 windows = 2 hours
six = [[0, 0, 1], [0, 0, -1], [1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0]]   # the 6 calibration positions
noise = np.random.default_rng(0).normal(0, 0.003, (6 * 500, 3))
calib = np.repeat(np.array(six, float), 500, axis=0) + noise   # 10 s each, in the watch's own frame
watch_frame = np.concatenate([calib, np.concatenate(model_frame) @ R_hidden])   # model = R w  ->  w = R^T model
raw_g = np.round((watch_frame * hidden_gain + hidden_offset) * 2048) / 2048   # the chip's whole counts, in g

fake = Path('processed/fake_watch_recording.csv')
n = len(raw_g)
with open(fake, 'w', newline='') as out:                         # the layout record.py writes
    out.write('# label: rehearsal\n# converted with accel_lsb_per_g=2048.0, gyro_lsb_per_dps=16.4\n')
    out.write('# sensor: MPU-6050, raw int16 counts, no conversion on device\n# accel_lsb_per_g: 2048\n')
    out.write('# columns: sample_index,micros,ax,ay,az,gx,gy,gz\n')
    out.write('sample_index,micros,ax_g,ay_g,az_g,gx_dps,gy_dps,gz_dps\n')
    pd.DataFrame({'i': np.arange(n), 'us': np.arange(n) * 20000, 'ax': raw_g[:, 0], 'ay': raw_g[:, 1], 'az': raw_g[:, 2],
                  'gx': 0.0, 'gy': 0.0, 'gz': 0.0}).to_csv(out, header=False, index=False, float_format='%.5f')
print(f'{fake} written: {n:,} samples ({n / 50 / 3600:.1f} h)')

# ----- the pipeline, from the file alone -----
acc, index = W.load_watch_csv(fake)
offset, gain, rep = W.fit_calibration(W.still_blocks(acc))         # 1. calibrate (uses the 6 positions + still wear)
acc_cal = (acc - offset) * gain
face_axis = int(np.abs(acc_cal[:500].mean(axis=0)).argmax())        # 2. face-up check: the first 10 s lie face up
windows = W.watch_windows(acc_cal[6 * 500:], index[6 * 500:])
uw, mw = W.window_summary(windows)
cands = W.z_kept() if face_axis == 2 else None
pick, R_found, _ = W.choose_orientation(uw, mw, reference, cands)   # 3. orientation
print('calibration:', {k: (round(v, 1) if isinstance(v, float) else v) for k, v in rep.items()})
print('offset found', offset.round(3), ' hidden', hidden_offset)
print('face-up axis:', 'xyz'[face_axis])
print('orientation found:', W.ORIENTATIONS[pick][0], '| hidden:', hidden, '| correct:', np.array_equal(R_found, R_hidden))
rehearsal = {'offset error': np.abs(offset - hidden_offset).max(), 'orientation correct': np.array_equal(R_found, R_hidden)}

processed\fake_watch_recording.csv written: 1,803,000 samples (10.0 h)


calibration: {'still blocks': 565, 'all axes covered': True, 'error before (mg)': np.float64(134.0), 'error after (mg)': np.float64(6.9), 'used': True}
offset found [ 0.013 -0.021  0.22 ]  hidden [ 0.01 -0.02  0.23]
face-up axis: z
orientation found: model x, y, z = watch -y, +x, +z | hidden: model x, y, z = watch -y, +x, +z | correct: True


## 6. The procedure for the watch

**Once per watch design and wrist** (the mounting doesn't change between wearers, but left vs right wrist does):

1. **Calibration positions (about 2 minutes).** Flash the logger, start `record.py`, and hold the watch still for 10 s in each of 6 positions: face up, face down, and each of the 4 edges down. Start with **face up**. Repeat a few times a year; calibration drifts
2. **Face-up check (from step 1).** Whichever raw axis reads about ±1 g while face up goes through the face
3. **Normal daytime wear: 3+ hours from 3–5 people** (or several days from one), on the wrist SilverGuard will be worn on. Ordinary life, no script. Long recordings need the Wi-Fi recorder rather than a USB cable
4. **Run the tool** (as in section 5): `load_watch_csv` → `still_blocks` + `fit_calibration` → `watch_windows` + `window_summary` → `choose_orientation` (16 candidates if the face axis is z)
5. **Put the result in the firmware**, applied to every sample before the models:

```c
// from watch_alignment.py (example values from the rehearsal; use your own)
const float   ACC_OFFSET_G[3] = {0.010f, -0.020f, 0.230f};   // subtract first (in the watch's own axes)
const float   ACC_GAIN[3]     = {1.010f, 0.990f, 0.971f};    // then multiply
const uint8_t MODEL_FROM[3]   = {1, 0, 2};                   // model x = watch y, model y = watch x, model z = watch z
const int8_t  MODEL_SIGN[3]   = {-1, +1, +1};                // model x = -watch y
const bool    MIRROR          = false;                       // true if the matrix is a mirror image (other wrist)

void to_model_frame(const float raw_g[3], const float raw_dps[3], float acc[3], float gyro[3]) {
  float cal[3];
  for (int i = 0; i < 3; i++) cal[i] = (raw_g[i] - ACC_OFFSET_G[i]) * ACC_GAIN[i];
  for (int i = 0; i < 3; i++) {
    acc[i]  = MODEL_SIGN[i] * cal[MODEL_FROM[i]];
    gyro[i] = MODEL_SIGN[i] * raw_dps[MODEL_FROM[i]] * (MIRROR ? -1 : 1);   // rotation flips under a mirror
  }
}
```

This replaces the single `acc_scale`. The fall model was trained on other data (FallAllD), so check whether it needs its own frame before applying this to it.

## 7. What this shows

**1. A wrong orientation costs accuracy, and how much depends on which axes get mixed up** (XGBoost + HMM, 30 Capture-24 test people, correct = 0.818 4-class / 0.790 binary kappa):

| Mix-up | 4-class | Still vs moving |
|---|---|---|
| x and z swapped | 0.787 (−0.03) | 0.739 (−0.05) |
| y and z swapped | 0.719 (−0.10) | 0.664 (−0.13) |
| worst of the 48 | 0.665 (−0.15) | 0.590 (−0.20) |
| y and z reversed (GOTOV's case) | 0.811 (−0.01) | 0.768 (−0.02) |

Some sign flips look harmless here because Capture-24's own people wore the sensor different ways. **Don't rely on that:** the same "y and z reversed" cost GOTOV 0.095 in still-vs-moving kappa (0.895 vs 0.990).

**2. The tool needs pooled data.** With 3 h from a single person and all 48 candidates, 13.5% of trials lose more than 0.05 kappa (worst 0.29). With 3 h from each of **5 people** and the **face-up check** (16 candidates), no trial loses more than 0.05 (worst 0.04).

**3. On a real second device (GOTOV) the tool gets it right:** it picks "y and z reversed", which scores 0.990 still-vs-moving (the best of all 48) and 0.825 3-class (0.002 behind the best). One person's hour alone agrees with the pooled choice for only 20 of 35 people, which is the same lesson as point 2.

**4. The firmware's single scale factor is not enough.** If the 1.23 g comes from an offset on one axis, dividing by 1.23 makes the watch read 0.63 g face down and 0.83 g with the arm hanging. Sphere fitting recovers per-axis offset and gain to within 0.001 g. Normal wear alone gave enough different still positions for only 10 of 30 people in 3 h, so **do the 6-position routine**.

**5. The full rehearsal works.** From a file in the exact `record.py` format, with hidden errors, the pipeline found the right orientation and the 0.23 g offset (within 0.01 g).

**Caveats**
- Not yet run on the real watch: this is ready for its first recordings
- "Capture-24's z goes through the device face" is supported only moderately (in still windows z carries most of gravity 49% of the time, x 36%). On GOTOV, the 16-candidate and 48-candidate searches agreed
- Only XGBoost + HMM was scored here. The CNN reads raw axes, so it is probably at least as sensitive

In [8]:
# every claim above, checked against the numbers
k = cost.set_index('orientation')
assert round(k.loc[W.ORIENTATIONS[W.IDENTITY][0], '4-class kappa'], 3) == 0.818 and round(right['binary kappa'], 3) == 0.790
expect = {'x and z swapped': (0.787, 0.739), 'y and z swapped': (0.719, 0.664), 'y and z reversed (GOTOV)': (0.811, 0.768)}
for name, (k4, kb) in expect.items():
    assert (round(k.loc[named[name], '4-class kappa'], 3), round(k.loc[named[name], 'binary kappa'], 3)) == (k4, kb), name
assert round(cost['4-class kappa'].min(), 3) == 0.665 and round(cost.nsmallest(1, '4-class kappa')['binary kappa'].iloc[0], 3) == 0.590
gotov_cost = gotov_truth.loc[best_all, 'binary kappa'] - gotov_truth.loc[W.IDENTITY, 'binary kappa']
assert round(gotov_cost, 3) == 0.095
s = summary
assert round(s.loc[('all 48', 1), 'share_losing_over_005'], 3) == 0.135 and round(s.loc[('all 48', 1), 'worst_loss'], 2) == 0.29
assert s.loc[('z kept (16)', 5), 'share_losing_over_005'] == 0 and round(s.loc[('z kept (16)', 5), 'worst_loss'], 2) == 0.04
assert W.ORIENTATIONS[best_all][0] == W.ORIENTATIONS[best_z][0] == 'model x, y, z = watch +x, -y, -z'
assert gotov_truth['binary kappa'].idxmax() == best_all and round(gotov_truth.loc[best_all, '3-class kappa'], 3) == 0.825
assert round(gotov_truth['3-class kappa'].max() - gotov_truth.loc[best_all, '3-class kappa'], 3) == 0.002
assert sum(agree) == 20 and len(agree) == 35
assert cal_check['offset error (g)'] < 0.001 and cal_check['gain error'] < 0.001
assert sum(covered) == 10
assert rehearsal['orientation correct'] and rehearsal['offset error'] < 0.015
assert axis_with_gravity.round(2).tolist() == [0.36, 0.15, 0.49]
print('every claim above matches the numbers')

every claim above matches the numbers
